# OpsPilot Ticket Intelligence Walkthrough

This notebook is a portfolio walkthrough for the English-only OpsPilot Ticket Intelligence baseline. The source of truth is the script pipeline, not the notebook.

Canonical flow:

1. `python scripts/build_ticket_dataset.py`
2. `python ml/ticket_intelligence/create_splits.py --input ../../data/processed/tickets_en_normalized.csv`
3. `python ml/ticket_intelligence/train_baseline.py`
4. `python ml/ticket_intelligence/evaluate.py`
5. `python ml/ticket_intelligence/feature_inspection.py`
6. `python ml/ticket_intelligence/error_analysis.py`
7. `python ml/ticket_intelligence/threshold_sweep.py`
8. `python ml/ticket_intelligence/predict.py`

## Project Goal

OpsPilot AI is an enterprise-style operations copilot. This vertical slice classifies support tickets, predicts priority, estimates escalation risk, and routes uncertain or high-risk tickets to human review.

The model does not make final customer-facing, refund, legal, compliance, or account decisions.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

VERTICAL_ROOT = Path.cwd()
if not (VERTICAL_ROOT / "ml" / "ticket_intelligence").exists():
    VERTICAL_ROOT = VERTICAL_ROOT.parent

PROJECT_ROOT = VERTICAL_ROOT.parents[1]
DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "tickets_en_normalized.csv"
TRAIN_PATH = VERTICAL_ROOT / "data" / "processed" / "train.csv"
VAL_PATH = VERTICAL_ROOT / "data" / "processed" / "val.csv"
TEST_PATH = VERTICAL_ROOT / "data" / "processed" / "test.csv"
OUTPUT_DIR = VERTICAL_ROOT / "ml" / "ticket_intelligence" / "outputs"
ARTIFACT_DIR = VERTICAL_ROOT / "ml" / "ticket_intelligence" / "artifacts"

print("Vertical root:", VERTICAL_ROOT)
print("Dataset exists:", DATASET_PATH.exists(), DATASET_PATH)
print("Train/val/test exist:", TRAIN_PATH.exists(), VAL_PATH.exists(), TEST_PATH.exists())

## Dataset Choice

The v1 dataset is `Tobi-Bueck/customer-support-tickets`, a public labeled customer-support ticket dataset. It is better aligned with OpsPilot than chatbot-intent-only datasets because it includes queue/category, priority, language, tags, ticket type, and support answers.

Version 1 is English-only to keep evaluation cleaner and TF-IDF features interpretable.

In [ ]:
dataset = pd.read_csv(DATASET_PATH)
print(dataset.shape)
display(dataset.head())
print("Language distribution")
display(dataset["language"].value_counts())
print("Category distribution")
display(dataset["true_category"].value_counts())
print("Priority distribution")
display(dataset["true_priority"].value_counts())

## Fixed Splits

The model uses fixed train/validation/test splits with `random_state=42` and category stratification where possible.

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

split_summary = pd.DataFrame([
    {"split":"train", "rows":len(train_df)},
    {"split":"val", "rows":len(val_df)},
    {"split":"test", "rows":len(test_df)},
])
display(split_summary)

display(train_df["true_category"].value_counts().rename("train_category_count").to_frame())
display(train_df["true_priority"].value_counts().rename("train_priority_count").to_frame())

## Baseline Metrics

The baseline is TF-IDF plus Logistic Regression for category and priority. These are honest baseline scores, not tuned or inflated demo numbers.

In [ ]:
metrics = json.loads((OUTPUT_DIR / "metrics.json").read_text())
summary = pd.DataFrame([
    {"target":"category", **{k: metrics["category"][k] for k in ["accuracy", "macro_f1", "weighted_f1"]}},
    {"target":"priority", **{k: metrics["priority"][k] for k in ["accuracy", "macro_f1", "weighted_f1"]}},
])
display(summary)

## Confusion Matrices

In [ ]:
display(Image(filename=str(OUTPUT_DIR / "category_confusion_matrix.png")))
display(Image(filename=str(OUTPUT_DIR / "priority_confusion_matrix.png")))

## Feature Inspection

Top positive features make the sparse baseline more explainable.

In [ ]:
display(pd.read_csv(OUTPUT_DIR / "top_features_by_category.csv").head(30))
display(pd.read_csv(OUTPUT_DIR / "top_features_by_priority.csv").head(30))

## Error Analysis

The error file tags likely reasons such as low confidence, overlapping labels, multiple intents, and priority depending on metadata.

In [ ]:
errors = pd.read_csv(OUTPUT_DIR / "error_analysis.csv")
print(errors.shape)
display(errors["error_type"].value_counts().to_frame())
display(errors.head(20))

## Threshold Sweep

This shows the operational tradeoff: stricter confidence thresholds reduce auto-triage volume but improve accuracy among auto-triaged tickets.

In [ ]:
thresholds = pd.read_csv(OUTPUT_DIR / "threshold_sweep.csv")
display(thresholds)

## Structured Prediction Demo

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "ml/ticket_intelligence/predict.py", "--ticket-id", "notebook-demo-001", "--no-save"],
    cwd=VERTICAL_ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Limitations And Next Steps

- Category labels overlap across technical support, product support, IT support, and customer service.
- Priority is text-only and should later include metadata such as SLA, customer tier, recurrence, and operational impact.
- Confidence is not calibrated yet.
- Escalation risk is rule-based and should be validated against real escalation outcomes.
- Multilingual support is future work with language-specific models or multilingual transformers such as XLM-R.
- Future product work can integrate this pipeline with FastAPI ticket creation and human review tables.